# EXP-004: Generate Doc-to-LoRA Adapters

Generate 8 per-document LoRA adapters using the SakanaAI Doc-to-LoRA hypernetwork.

**Requirements:** T4 GPU (Colab free tier) or better.

**Output:** `d2l_adapters.zip` — download and extract to `models/d2l/` locally.

In [ ]:
# 1. HuggingFace login (Gemma is gated — requires accepting Google's license)
# Get your token at https://huggingface.co/settings/tokens
# Accept Gemma license at https://huggingface.co/google/gemma-2-2b-it
from huggingface_hub import login
login()

In [ ]:
# 2. Install dependencies
# Pin versions matching the working local environment.
# --no-deps on D2L avoids pulling vllm/deepspeed.
# flash-attn is NOT compatible with T4 (Turing SM75, needs Ampere SM80+).
# We patch D2L's perceiver to use eager attention instead.
!pip install -q --no-deps git+https://github.com/SakanaAI/doc-to-lora.git
!pip install -q \
    transformers==4.51.3 \
    huggingface-hub==0.36.2 \
    peft==0.18.1 \
    accelerate==1.6.0 \
    bitsandbytes==0.49.2 \
    safetensors==0.7.0 \
    einops==0.8.2 \
    jaxtyping==0.3.9 \
    opt-einsum \
    pymupdf

In [ ]:
# 3. Download D2L checkpoint + pre-cache Gemma base model
!huggingface-cli download SakanaAI/doc-to-lora \
    --local-dir trained_d2l \
    --include "gemma_demo/checkpoint-80000/*"

# Pre-download Gemma so _init_model's local_files_only=True doesn't fail
from huggingface_hub import snapshot_download
snapshot_download("google/gemma-2-2b-it")
print("Checkpoint + Gemma cached.")

In [ ]:
# 4. Upload corpus
# Upload corpus.zip containing the 8 PDF files from data/corpus/
from google.colab import files
import os, shutil, zipfile, glob

os.makedirs("corpus", exist_ok=True)
print("Upload corpus.zip (data/corpus/ zipped) or individual .pdf files:")
uploaded = files.upload()

for fname in uploaded:
    if fname.endswith(".zip"):
        with zipfile.ZipFile(fname) as zf:
            zf.extractall("_tmp_corpus")
        for pdf in glob.glob("_tmp_corpus/**/*.pdf", recursive=True):
            shutil.copy(pdf, f"corpus/{os.path.basename(pdf)}")
        shutil.rmtree("_tmp_corpus")
    elif fname.endswith(".pdf"):
        shutil.copy(fname, f"corpus/{fname}")

corpus_files = sorted(f for f in os.listdir("corpus") if f.endswith(".pdf"))
assert len(corpus_files) > 0, "No .pdf files found in corpus/ -- check your upload"
print(f"Found {len(corpus_files)} corpus PDFs: {corpus_files}")

In [ ]:
# 5. Load D2L model
#
# D2L requires flash_attention_2 (hard assert), but T4 is Turing (SM75) and
# flash-attn needs Ampere (SM80+). We patch 5 incompatibilities:
#   1) Missing flash-attn utility funcs in newer transformers
#   2) use_cache=None rejected by newer Gemma2
#   3) flash_attention_2 ImportError → fallback to eager
#   4) Missing "eager" key in perceiver attention class registry
#   5) Hard assert self._use_flash_attention_2 in PerceiverResampler.__init__
#   6) Skip redundant _init_model (double-load OOM fix)

import gc
import torch
from transformers import BitsAndBytesConfig

# --- Shim 1: flash-attn utils removed in newer transformers ---
import transformers.utils as _tu
if not hasattr(_tu, "is_flash_attn_greater_or_equal_2_10"):
    _tu.is_flash_attn_greater_or_equal_2_10 = lambda: False
if not hasattr(_tu, "is_flash_attn_2_available"):
    _tu.is_flash_attn_2_available = lambda: False

# --- Shim 2: strip use_cache=None from from_pretrained ---
from transformers import AutoModelForCausalLM
_orig_auto_fp = AutoModelForCausalLM.from_pretrained
def _stripped_from_pretrained(*args, **kwargs):
    kwargs.pop("use_cache", None)
    return _orig_auto_fp(*args, **kwargs)
AutoModelForCausalLM.from_pretrained = _stripped_from_pretrained

# --- Shim 3: flash ImportError → eager fallback ---
from transformers.modeling_utils import PreTrainedModel
_orig_flash_check = PreTrainedModel._check_and_enable_flash_attn_2
@classmethod
def _safe_flash_check(cls, config, *args, **kwargs):
    try:
        return _orig_flash_check.__func__(cls, config, *args, **kwargs)
    except ImportError:
        config._attn_implementation = "eager"
        return config
PreTrainedModel._check_and_enable_flash_attn_2 = _safe_flash_check

# --- Shim 4: register eager attention class if missing ---
from ctx_to_lora.modeling import idefics2 as _idefics2_mod
_attn_cls_dict = _idefics2_mod.IDEFICS2_PERCEIVER_ATTENTION_CLASSES
if "eager" not in _attn_cls_dict:
    _attn_cls_dict["eager"] = _idefics2_mod.Idefics2PerceiverAttention

# --- Shim 5: bypass hard assert in PerceiverResampler.__init__ ---
# The assert is the last line of __init__; model is fully constructed by then.
_orig_resampler_init = _idefics2_mod.Idefics2PerceiverResampler.__init__
def _no_assert_resampler_init(self, config, *args, **kwargs):
    try:
        _orig_resampler_init(self, config, *args, **kwargs)
    except AssertionError:
        self._use_flash_attention_2 = False
_idefics2_mod.Idefics2PerceiverResampler.__init__ = _no_assert_resampler_init

# --- Load model ---
from ctx_to_lora.modeling.hypernet import ModulatedPretrainedModel

# --- Shim 6: skip redundant _init_model in load_state_dict ---
_original_load_state_dict = ModulatedPretrainedModel.load_state_dict
def _lean_load_state_dict(self, state_dict, *_args, **_kwargs):
    self.base_model_name_or_path = state_dict.pop("base_model_name_or_path")
    self.hypernet_config = state_dict.pop("hypernet_config")
    self.ctx_encoder_args = state_dict.pop("ctx_encoder_args")
    if self.base_model_name_or_path != self.base_model.name_or_path:
        raise ValueError(
            f"Base model mismatch: model={self.base_model.name_or_path}, "
            f"checkpoint={self.base_model_name_or_path}"
        )
    prefix = "_orig_mod."
    for k in list(state_dict.keys()):
        if k.startswith(prefix):
            state_dict[k[len(prefix):]] = state_dict.pop(k)
    result = self.hypernet.load_state_dict(state_dict, strict=True)
    print(f"load result: {result}")
    return result

checkpoint_path = "trained_d2l/gemma_demo/checkpoint-80000/pytorch_model.bin"
state_dict = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

ModulatedPretrainedModel.load_state_dict = _lean_load_state_dict
try:
    model = ModulatedPretrainedModel.from_state_dict(
        state_dict,
        train=False,
        base_model_kwargs={
            "quantization_config": bnb_config,
            "device_map": "auto",
            "local_files_only": True,
            "attn_implementation": "eager",
        },
        use_flash_attn=False,
        use_sequence_packing=False,
    )
finally:
    ModulatedPretrainedModel.load_state_dict = _original_load_state_dict
    AutoModelForCausalLM.from_pretrained = _orig_auto_fp
    PreTrainedModel._check_and_enable_flash_attn_2 = _orig_flash_check
    _idefics2_mod.Idefics2PerceiverResampler.__init__ = _orig_resampler_init

model.reset()
gc.collect()
torch.cuda.empty_cache()

print("Model loaded.")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.memory_allocated()/1024**3:.1f} GiB allocated")

In [ ]:
# 6. Generate adapters for each document
import time
import fitz  # PyMuPDF
from pathlib import Path
from copy import deepcopy
from safetensors.torch import save_file as save_safetensors_file
from ctx_to_lora.utils import generated_lora_to_state_dict, get_lora_module_names

def extract_pdf_text(pdf_path: Path) -> str:
    """Extract full text from PDF, joining pages with double newlines."""
    doc = fitz.open(pdf_path)
    try:
        pages = [page.get_text("text").strip() for page in doc]
    finally:
        doc.close()
    return "\n\n".join(p for p in pages if p).strip()

def normalize_lora_b_orientation(state_dict: dict) -> dict:
    """Transpose lora_B from D2L's (r, d_out) to PEFT's (d_out, r) if needed."""
    normalized = {k: v.detach().cpu().contiguous() for k, v in state_dict.items()}
    for key, value in list(normalized.items()):
        if ".lora_B." not in key or value.ndim != 2:
            continue
        prefix = key.split(".lora_B.")[0]
        key_a = f"{prefix}.lora_A.weight"
        if key_a not in normalized:
            continue
        rank = normalized[key_a].shape[0]
        if value.shape[0] == rank:
            normalized[key] = value.transpose(0, 1).contiguous()
    return normalized

corpus_dir = Path("corpus")
output_root = Path("adapters")

corpus_files = sorted(corpus_dir.glob("*.pdf"))
assert len(corpus_files) > 0, "No .pdf files in corpus/"
print(f"Processing {len(corpus_files)} documents...")

# Match local pipeline: model.peft_config may be a LoraConfig, not a dict
peft_config = model.peft_config
if isinstance(peft_config, dict):
    peft_config = list(peft_config.values())[0]
target_modules = sorted(str(m) for m in (peft_config.target_modules or []))
layer_indices = sorted(int(idx) for idx in model.hypernet.layer_indices)
module_names = get_lora_module_names(
    model.base_model, target_modules=target_modules, layer_indices=layer_indices
)

for doc_idx, pdf_path in enumerate(corpus_files, start=1):
    doc_text = extract_pdf_text(pdf_path)
    word_count = len(doc_text.split())
    print(f"\n[{doc_idx}/{len(corpus_files)}] {pdf_path.name} ({word_count} words)")

    gc.collect()
    torch.cuda.empty_cache()
    start = time.perf_counter()
    model.internalize(doc_text)
    gen_time = time.perf_counter() - start
    print(f"  Internalized in {gen_time:.1f}s")

    generated_sd = generated_lora_to_state_dict(
        model.generated_loras, module_names, target_modules, layer_indices
    )

    adapter_dir = output_root / f"doc{doc_idx}"
    adapter_dir.mkdir(parents=True, exist_ok=True)

    # Normalize lora_B orientation before saving (D2L may emit (r, d_out))
    export_sd = normalize_lora_b_orientation(generated_sd)
    save_safetensors_file(export_sd, str(adapter_dir / "adapter_model.safetensors"))

    export_config = deepcopy(peft_config)
    export_config.base_model_name_or_path = model.base_model.name_or_path
    export_config.inference_mode = True
    export_config.save_pretrained(str(adapter_dir))

    n_tensors = len(export_sd)
    total_bytes = sum(f.stat().st_size for f in adapter_dir.iterdir())
    print(f"  Saved to {adapter_dir} ({n_tensors} tensors, {total_bytes/1024:.0f} KB)")

    model.reset()

print(f"\nDone! All {len(corpus_files)} adapters saved to {output_root}/")

In [ ]:
# 7. Zip and download
import shutil
from google.colab import files
# Archive so zip root contains doc1/, doc2/, ... directly
shutil.make_archive("d2l_adapters", "zip", "adapters")
files.download("d2l_adapters.zip")
print("Extract: unzip d2l_adapters.zip -d models/d2l/")